# 📓 Semana 8 · Dia 1 — Tuning Spark: AQE, broadcast join e cache estratégico

**Curso**: Especialista Databricks — Engenharia de Dados → GenAI → Agentes

| Campo | Valor |
|---|---|
| **Plano** | ✅ Free Edition |
| **Tempo estimado** | 2h |
| **Certificação alvo** | DEP (performance) |
| **Pré-requisitos** | SQL e Python básicos · notebooks anteriores do curso |
| **Entregável do dia** | Benchmark antes/depois documentado |

---


## 📖 Teoria — O caminho do tuning (na ordem certa)

1. **Plano físico** (explain) — ver onde está o custo
2. **Evitar shuffle** — broadcast join, boas chaves
3. **AQE (Adaptive Query Execution)** — otimiza em runtime
4. **Cache** — só para reuso real
5. **Storage** — Liquid Clustering, OPTIMIZE

> 🎯 **Dica de prova (DEP)**: tuning NÃO começa com `spark.conf`. Começa com o plano físico e a eliminação de shuffle.


## 📖 Teoria — AQE — Adaptive Query Execution

O AQE ajusta o plano **durante a execução**: reduz partições de shuffle quando há poucos dados, converte sort-merge em broadcast quando a tabela ficou pequena, e corrige skew (partições desbalanceadas).

Configurações (raro mexer; entender é o que cai na prova):
```
spark.sql.adaptive.enabled=true
spark.sql.adaptive.coalescePartitions.enabled=true
spark.sql.adaptive.skewJoin.enabled=true
```


### 💻 Na prática — Encontrando o custo no explain

Compare o plano de um join e veja onde está o shuffle.


In [ ]:
# Dataset de teste
df = spark.table("workspace.prata.fato_vendas")
dim = spark.table("workspace.prata.dim_produto")
print("Fato:", df.count(), "| Dim:", dim.count())

In [ ]:
# Plano físico do join (identifique Exchange/Shuffle)
plano = df.join(dim, "sk_produto", "left")
print(plano.explain("formatted"))
print("Procure: Exchange (shuffle) e SortMergeJoin vs BroadcastHashJoin")

### 💻 Na prática — Forçando broadcast

Uma dimensão pequena deve ser broadcast — economiza shuffle.


In [ ]:
# Sem hint (pode shuffle)
t_sem = df.join(dim, "sk_produto", "left").count()
# Com hint broadcast
t_com = df.join(dim.hint("broadcast"), "sk_produto", "left").count()
print("Resultado igual:", t_sem == t_com)
print("Veja no explain: BroadcastExchange eliminou o shuffle.")

### 💻 Na prática — Cache estratégico

Meça o efeito de cache em reuso real.


In [ ]:
# Sem cache: recalcula a cada uso
import time
base = spark.table("workspace.prata.fato_vendas").filter("Country = 'United Kingdom'")
t0 = time.time(); base.groupBy("sk_produto").count().count(); t1 = time.time()
t2 = time.time(); base.groupBy("Country").count().count(); t3 = time.time()
print(f"Sem cache: {t1-t0:.2f}s + {t3-t2:.2f}s (recalcula 2x)")

In [ ]:
# Com cache (uma vez em memória)
base_cached = base.cache()
base_cached.count()  # materializa
t0 = time.time(); base_cached.groupBy("sk_produto").count().count(); t1 = time.time()
t2 = time.time(); base_cached.groupBy("Country").count().count(); t3 = time.time()
print(f"Com cache: {t1-t0:.2f}s + {t3-t2:.2f}s (mais rápido no reuso)")
base_cached.unpersist()

> 🎯 **Dica de prova**: Pergunta DEP típica: 'o que o AQE faz?' → coalesce de partições, conversão para broadcast, correção de skew. 'Como eliminar shuffle?' → broadcast join em tabela pequena.


## 🎯 Exercícios de fixação

**1.** Explique por que broadcast join elimina o shuffle.

**2.** Quais 3 otimizações o AQE faz em runtime?

**3.** Quando cache NÃO ajuda?


> Tente resolver **antes** de olhar o gabarito no final do notebook.


## 🗝️ Gabarito comentado

**1.** Broadcast

A tabela pequena é copiada para cada executor — o join roda localmente, sem mover linhas entre partições (sem Exchange).

**2.** AQE

1) coalesce partições de shuffle (une partições pequenas); 2) converte sort-merge em broadcast se o tamanho permitir; 3) corrige skew de join (divide partição quente).

**3.** Cache sem reuso

Se o DataFrame é usado 1x, cache só ocupa memória e adiciona trabalho (o primeiro count materializa na mesma velocidade). Cache paga quando o mesmo resultado é consumido várias vezes.



## ✅ Checklist de fechamento

- [ ] Rodei todas as células do notebook do início ao fim sem erros.
- [ ] Consigo explicar os conceitos de hoje em 3 frases (sem olhar o material).
- [ ] Fiz os exercícios e conferi o gabarito.
- [ ] Anotei as dúvidas que preciso revisar.

---
*Próximo passo: siga para o notebook seguinte do plano do curso.*